# 04 — Risk Measures: VaR, CVaR & Exceedance Backtesting

**What this notebook does:**

VaR and CVaR were already computed **per-window** inside the rolling loop (notebook 02), using the per-window NIG distribution and the affine transform:

$$\text{VaR}_\alpha = \mu_{t+1} + \sigma_{t+1} \cdot F^{-1}_{\text{NIG}_t}(1-\alpha)$$

This notebook loads those pre-computed risk measures and performs backtesting:

1. VaR exceedance counting with binomial $p$-values 
2. Color-coded results (green/red/blue per professor's convention)
3. VaR vs actual returns visualisation
4. CVaR summary (Expected Shortfall — required by Basel III)
5. Christoffersen independence test on hit sequences

In [1]:
# Imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "iframe"

import warnings
warnings.filterwarnings("ignore")

from src.data_utils import load_processed, save_processed
from src.assessment import (
    count_exceedances, binomial_pvalue, pvalue_color,
    christoffersen_test,
)

print("Imports OK")


Imports OK


In [2]:
# Load pre-computed risk measures from notebook 02

var99 = load_processed("var99.parquet")
var95 = load_processed("var95.parquet")
cvar99 = load_processed("cvar99.parquet")
cvar95 = load_processed("cvar95.parquet")

TICKERS = list(var99.columns)

# Load full rolling results for actual returns
all_results = {}
for ticker in TICKERS:
    safe = ticker.replace("^", "").replace("=", "_")
    all_results[ticker] = load_processed(f"rolling_{safe}.parquet")

N = len(var99)  # assessment window size

print(f"Loaded: {len(TICKERS)} assets, {N} prediction dates")
print(f"Date range: {var99.index[0].date()} → {var99.index[-1].date()}")
print(f"Tickers: {TICKERS}")

# Quick sanity check
assert (var99 < 0).all().all(), "VaR 99% should all be negative"
assert (var95 < 0).all().all(), "VaR 95% should all be negative"
print("\nAll VaR values negative: PASS")

Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\var95.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar99.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar95.parquet  shape=(1000, 5)
Loaded from C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model

---
## 1. VaR Exceedance Backtesting

An exceedance occurs when the actual return falls below VaR (Kaufman slide 28):

$$e_t = \begin{cases} 1 & \text{if } r_t < \text{VaR}^t_\alpha \\ 0 & \text{otherwise} \end{cases}$$

The total exceedance count $E = \sum e_t \sim \text{Binomial}(n, 1-\alpha)$. We compute a two-sided binomial $p$-value.

**Color coding:**
- **Green:** $p \geq 0.05$ — model passes
- **Red:** $p < 0.05$ and exceedances > expected — risk understated
- **Blue:** $p < 0.05$ and exceedances < expected — risk overstated

In [4]:
# Count exceedances and compute binomial p-values

LEVELS = [0.99, 0.95]
var_data = {0.99: var99, 0.95: var95}

results_table = []

for ticker in TICKERS:
    actual = all_results[ticker]["return"].values

    for level in LEVELS:
        var_s    = var_data[level][ticker].values
        exceed   = count_exceedances(actual, var_s)
        expected = N * (1 - level)
        pval     = binomial_pvalue(N, exceed, level)
        color    = pvalue_color(pval, exceed, expected)

        results_table.append({
            "Ticker":      ticker,
            "VaR level":   f"{int(level*100)}%",
            "Expected":    int(expected),
            "Exceedances": exceed,
            "Excess":      exceed - int(expected),
            "p-value":     round(pval, 4),
            "Result":      color.upper(),
        })

results_df = pd.DataFrame(results_table)
print("VaR Exceedance Backtesting Results\n")
print(results_df.to_string(index=False))

VaR Exceedance Backtesting Results

  Ticker VaR level  Expected  Exceedances  Excess  p-value Result
    AAPL       99%        10           14       4   0.2006  GREEN
    AAPL       95%        50           59       9   0.1912  GREEN
EURUSD=X       99%        10           12       2   0.5215  GREEN
EURUSD=X       95%        50           52       2   0.7714  GREEN
     GLD       99%        10           12       2   0.5215  GREEN
     GLD       95%        50           56       6   0.3834  GREEN
     TIP       99%        10           14       4   0.2006  GREEN
     TIP       95%        50           50       0   1.0000  GREEN
   ^GSPC       99%        10           17       7   0.0365    RED
   ^GSPC       95%        50           60      10   0.1465  GREEN


In [5]:
# Color-coded p-value table

color_map = {"GREEN": "#2dc653", "RED": "#e63946", "BLUE": "#457b9d"}
font_map  = {"GREEN": "#144a23", "RED": "#5c0a10", "BLUE": "#0d2b40"}

def make_pvalue_table(df, title):
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["Ticker", "Expected", "Exceedances",
                    "Excess", "p-value", "Result"],
            fill_color="#2c3e50",
            font=dict(color="white", size=12),
            align="center", height=32,
        ),
        cells=dict(
            values=[
                df.index.tolist(),
                df["Expected"].tolist(),
                df["Exceedances"].tolist(),
                df["Excess"].tolist(),
                df["p-value"].tolist(),
                df["Result"].tolist(),
            ],
            fill_color=[
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                ["#f8f9fa"] * len(df),
                [color_map[r] for r in df["Result"]],
            ],
            font=dict(
                color=[
                    ["#2c3e50"] * len(df)] * 5 + [
                    [font_map[r] for r in df["Result"]],
                ],
                size=12,
            ),
            align="center", height=28,
        ),
    )])
    fig.update_layout(
        title=title, template="plotly_white",
        height=260, margin=dict(t=50, b=10, l=10, r=10),
    )
    return fig

In [6]:
# VaR 99% backtesting table

var99_df = results_df[results_df["VaR level"] == "99%"].set_index("Ticker")
fig99 = make_pvalue_table(var99_df, "VaR 99% Backtesting Results")
fig99.write_html("../outputs/figures/04_pvalue_table_99.html")
fig99.show()
print("Table saved")


Table saved


In [7]:
# VaR 95% backtesting table

var95_df = results_df[results_df["VaR level"] == "95%"].set_index("Ticker")
fig95 = make_pvalue_table(var95_df, "VaR 95% Backtesting Results")
fig95.write_html("../outputs/figures/04_pvalue_table_95.html")
fig95.show()
print("Table saved")

Table saved


---
## 2. VaR vs Actual Returns

In [8]:
# VaR vs actual returns for all assets

colors_assets = dict(zip(TICKERS, px.colors.qualitative.Plotly))

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=TICKERS,
    vertical_spacing=0.06,
)

for i, ticker in enumerate(TICKERS):
    actual = all_results[ticker]["return"]
    v99    = var99[ticker]
    v95    = var95[ticker]
    exceed_mask = actual.values < v99.values

    # Actual returns
    fig.add_trace(go.Scatter(
        x=actual.index, y=actual.values,
        mode="lines", name=f"{ticker} returns",
        line=dict(color=colors_assets[ticker], width=0.7),
        showlegend=False,
    ), row=i+1, col=1)

    # VaR 99%
    fig.add_trace(go.Scatter(
        x=v99.index, y=v99.values,
        mode="lines",
        name="VaR 99%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#e63946", width=1.5),
    ), row=i+1, col=1)

    # VaR 95%
    fig.add_trace(go.Scatter(
        x=v95.index, y=v95.values,
        mode="lines",
        name="VaR 95%" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#c47000", width=1.2, dash="dot"),
    ), row=i+1, col=1)

    # Exceedances
    exc_dates  = actual.index[exceed_mask]
    exc_values = actual.values[exceed_mask]
    fig.add_trace(go.Scatter(
        x=exc_dates, y=exc_values,
        mode="markers",
        name="Exceedance" if i == 0 else None,
        showlegend=(i == 0),
        marker=dict(color="#040720", size=5, symbol="x"),
    ), row=i+1, col=1)

    fig.update_yaxes(title_text="Log return", row=i+1, col=1)

fig.update_layout(
    title="Actual Returns vs VaR 99% and VaR 95% — All Assets",
    height=1000, template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.04),
)
fig.write_html("../outputs/figures/04_var_vs_returns.html")
fig.show()
print("Figure saved")

Figure saved


---
## 3. Christoffersen Independence Test

Tests whether VaR exceedances cluster (serial dependence) or are independent. Clustering indicates the GARCH model has not fully captured volatility dynamics.

$p > 0.05$ means exceedances are independent (model passes).

In [9]:
# Christoffersen independence test on hit sequences

christ_rows = []
for ticker in TICKERS:
    actual = all_results[ticker]["return"].values
    for level in LEVELS:
        var_s = var_data[level][ticker].values
        hits  = (actual < var_s).astype(int)
        result = christoffersen_test(hits)

        christ_rows.append({
            "Ticker": ticker,
            "VaR level": f"{int(level*100)}%",
            "Statistic": result["statistic"],
            "p-value": result["pvalue"],
            "Independent": "✓" if result["independent"] else "✗",
        })

christ_df = pd.DataFrame(christ_rows)
print("Christoffersen Independence Test\n")
print(christ_df.to_string(index=False))
print("\n✓ = exceedances are independent (p > 0.05)")
print("✗ = exceedances cluster — residual volatility dynamics remain")

Christoffersen Independence Test

  Ticker VaR level  Statistic  p-value Independent
    AAPL       99%     6.1763   0.0129           ✗
    AAPL       95%     0.6662   0.4144           ✓
EURUSD=X       99%     0.2918   0.5891           ✓
EURUSD=X       95%     5.7135   0.0168           ✗
     GLD       99%     0.2918   0.5891           ✓
     GLD       95%     2.3766   0.1232           ✓
     TIP       99%     0.3980   0.5281           ✓
     TIP       95%     0.1037   0.7475           ✓
   ^GSPC       99%     1.1211   0.2897           ✓
   ^GSPC       95%     4.6932   0.0303           ✗

✓ = exceedances are independent (p > 0.05)
✗ = exceedances cluster — residual volatility dynamics remain


---
## 4. CVaR (Expected Shortfall) Summary

CVaR = $E[R \mid R < \text{VaR}_\alpha]$ — the average loss in the worst scenarios. Required by Basel III as a supplement to VaR.

In [11]:
# CVaR summary

cvar_rows = []
for ticker in TICKERS:
    cvar_rows.append({
        "Ticker": ticker,
        "CVaR 99% (mean)": round(cvar99[ticker].mean() * 100, 4),
        "CVaR 95% (mean)": round(cvar95[ticker].mean() * 100, 4),
        "CVaR 99% (worst)": round(cvar99[ticker].min() * 100, 4),
        "CVaR 95% (worst)": round(cvar95[ticker].min() * 100, 4),
    })

cvar_summary = pd.DataFrame(cvar_rows).set_index("Ticker")
print("CVaR Summary (% log returns --> negative = loss threshold)\n")
display(cvar_summary)
print("\nCVaR is always more negative than VaR —")
print("it represents the average loss in the worst scenarios, not just the threshold.")

CVaR Summary (% log returns --> negative = loss threshold)



,CVaR 99% (mean),CVaR 95% (mean),CVaR 99% (worst),CVaR 95% (worst)
Ticker,,,,
AAPL,-5.2817,-3.6047,-31.5023,-19.2237
EURUSD=X,-1.4525,-1.0588,-3.5159,-2.3814
GLD,-2.8627,-2.0304,-9.7225,-7.0129
TIP,-1.1886,-0.8632,-2.8538,-2.0642
^GSPC,-3.1993,-2.2963,-18.5130,-11.9804



CVaR is always more negative than VaR —
it represents the average loss in the worst scenarios, not just the threshold.


---

In [12]:
# Save results

save_processed(results_df, "exceedance_results.parquet")
save_processed(christ_df, "christoffersen_results.parquet")
save_processed(cvar_summary.reset_index(), "cvar_summary.parquet")
print("Saved: exceedance_results, christoffersen_results, cvar_summary")

Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\exceedance_results.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\christoffersen_results.parquet
Saved to C:\Users\amosa\Documents\My Graduate School\SPRING 2026\Courses\AMS603_Risk Measures for Finance and Data Analysis\Projects\arma-garch-beta-risk-model\data\processed\cvar_summary.parquet
Saved: exceedance_results, christoffersen_results, cvar_summary


In [13]:
# Verification

print("Notebook 04 — Summary\n")
print(f"Assets: {len(TICKERS)}, Predictions: {N}")
print(f"\nVaR Exceedance Results:")
print(results_df[["Ticker", "VaR level", "Expected",
                   "Exceedances", "p-value", "Result"]].to_string(index=False))
print(f"\nChristoffersen Results:")
print(christ_df[["Ticker", "VaR level", "p-value",
                  "Independent"]].to_string(index=False))
print("\nProceed to notebook 05 (full assessment suite).")

Notebook 04 — Summary

Assets: 5, Predictions: 1000

VaR Exceedance Results:
  Ticker VaR level  Expected  Exceedances  p-value Result
    AAPL       99%        10           14   0.2006  GREEN
    AAPL       95%        50           59   0.1912  GREEN
EURUSD=X       99%        10           12   0.5215  GREEN
EURUSD=X       95%        50           52   0.7714  GREEN
     GLD       99%        10           12   0.5215  GREEN
     GLD       95%        50           56   0.3834  GREEN
     TIP       99%        10           14   0.2006  GREEN
     TIP       95%        50           50   1.0000  GREEN
   ^GSPC       99%        10           17   0.0365    RED
   ^GSPC       95%        50           60   0.1465  GREEN

Christoffersen Results:
  Ticker VaR level  p-value Independent
    AAPL       99%   0.0129           ✗
    AAPL       95%   0.4144           ✓
EURUSD=X       99%   0.5891           ✓
EURUSD=X       95%   0.0168           ✗
     GLD       99%   0.5891           ✓
     GLD       95%  

---